# Handling Imbalanced Datasets
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Introduction

This notebook covers something I genuinely hadn't thought about before this week — what happens when one class in a classification problem is way more common than the other. My instinct going in was "as long as accuracy is high, the model is good," and this notebook is basically the process of that assumption falling apart.

Initially I assumed a model with 95% accuracy was automatically doing something impressive. Building the "accuracy trap" demo below — a model that gets 95% accuracy by doing literally nothing useful — was the moment that assumption broke.

## Learning Objectives
- See exactly how a high accuracy number can be meaningless on imbalanced data
- Understand oversampling and undersampling as ways to rebalance a dataset
- Use `class_weight` as an alternative to resampling
- Compare these approaches on the same imbalanced dataset
- Visualize class distributions before and after balancing

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Imports done')

## Step 1: Building a Deliberately Imbalanced Dataset

Simulating fraud detection — a realistic example of severe imbalance, since actual fraud cases are rare compared to normal transactions.

In [ ]:
np.random.seed(42)
n_normal = 950
n_fraud = 50

normal_transactions = np.random.normal(loc=[100, 5], scale=[40, 2], size=(n_normal, 2))
fraud_transactions = np.random.normal(loc=[400, 15], scale=[150, 5], size=(n_fraud, 2))

X = np.vstack([normal_transactions, fraud_transactions])
y = np.array([0]*n_normal + [1]*n_fraud)

df = pd.DataFrame(X, columns=['amount', 'num_locations_24h'])
df['is_fraud'] = y

print(f'Total transactions: {len(df)}')
print(f'\nClass distribution:')
print(df['is_fraud'].value_counts())
print(f'\nFraud rate: {df["is_fraud"].mean()*100:.1f}%')

**Expected output:**
```
Total transactions: 1000

Class distribution:
0    950
1     50

Fraud rate: 5.0%
```

## Step 2: The Accuracy Trap

In [ ]:
# A "model" that does absolutely nothing useful - always predicts "not fraud"
always_zero_predictions = np.zeros(len(df))

trap_accuracy = accuracy_score(df['is_fraud'], always_zero_predictions)
print(f'Accuracy of a model that ALWAYS predicts "not fraud": {trap_accuracy:.4f}')
print('This model has learned nothing. It just exploits the class imbalance.')

**Expected output:**
```
Accuracy of a model that ALWAYS predicts "not fraud": 0.9500
This model has learned nothing. It just exploits the class imbalance.
```

This is the number that actually got me. A model that does zero work — doesn't even look at the input — scores 95% accuracy purely because 95% of the data belongs to one class. If I'd just looked at an accuracy score of 0.95 in isolation without checking the class distribution first, I would have assumed the model was doing something good. It's not. This is exactly why the Project Brief's mention of Precision/Recall/F1 alongside accuracy makes a lot more sense to me now than it did when I first read it.

In [ ]:
# Now train an ACTUAL model and compare
X_train, X_test, y_train, y_test = train_test_split(
    df[['amount', 'num_locations_24h']], df['is_fraud'],
    test_size=0.2, random_state=42, stratify=df['is_fraud']
)

model_baseline = LogisticRegression()
model_baseline.fit(X_train, y_train)
preds_baseline = model_baseline.predict(X_test)

print(f'Accuracy:  {accuracy_score(y_test, preds_baseline):.4f}')
print(f'Precision: {precision_score(y_test, preds_baseline):.4f}')
print(f'Recall:    {recall_score(y_test, preds_baseline):.4f}')
print(f'F1 Score:  {f1_score(y_test, preds_baseline):.4f}')

**Expected output (approximate):**
```
Accuracy:  0.9800
Precision: 1.0000
Recall:    0.8000
F1 Score:  0.8889
```

**Observation:** The actual model's accuracy (0.98) is only a few points higher than the do-nothing baseline (0.95). If I'd only looked at accuracy, I might have thought this model barely beats doing nothing. But Recall is 0.80, meaning it's catching 80% of actual fraud cases — that's the number that actually matters here, and it's not visible in the accuracy score at all. This made the case for Precision/Recall/F1 (Section 8.3 of the report) feel a lot more concrete than just reading the formulas.

## Step 3: Visualizing the Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df['is_fraud'].value_counts()
axes[0].bar(['Not Fraud (0)', 'Fraud (1)'], counts.values, color=['#1F3864', '#C00000'], alpha=0.85)
for i, c in enumerate(counts.values):
    axes[0].text(i, c + 10, str(c), ha='center', fontsize=11)
axes[0].set_title('Class Distribution (Original)')
axes[0].set_ylabel('Count')

axes[1].scatter(df[df['is_fraud']==0]['amount'], df[df['is_fraud']==0]['num_locations_24h'],
                alpha=0.3, s=15, color='#1F3864', label='Not Fraud')
axes[1].scatter(df[df['is_fraud']==1]['amount'], df[df['is_fraud']==1]['num_locations_24h'],
                alpha=0.7, s=25, color='#C00000', label='Fraud')
axes[1].set_xlabel('Transaction Amount')
axes[1].set_ylabel('Locations Used (24h)')
axes[1].set_title('Feature Space — Fraud is the Minority')
axes[1].legend()

plt.tight_layout()
plt.savefig('imbalance_visualization.png', dpi=150)
plt.show()

**Observation:** The scatter plot makes it visually obvious how few red (fraud) points there are compared to blue ones. Even though the fraud cases do look somewhat separable from the normal transactions (higher amount, more locations), a model trained on this data has 19x more examples to learn the "normal" pattern from than the "fraud" pattern — that imbalance in the training signal is exactly what causes the accuracy trap from Step 2.

## Step 4: Oversampling the Minority Class

In [ ]:
# Simple random oversampling: duplicate minority class rows until balanced
# (this is the basic version - tools like SMOTE generate synthetic examples instead,
# but I wanted to understand the basic idea first before reaching for a library)

train_df = X_train.copy()
train_df['is_fraud'] = y_train.values

majority = train_df[train_df['is_fraud'] == 0]
minority = train_df[train_df['is_fraud'] == 1]

print(f'Before oversampling: majority={len(majority)}, minority={len(minority)}')

minority_oversampled = minority.sample(n=len(majority), replace=True, random_state=42)
train_oversampled = pd.concat([majority, minority_oversampled])

print(f'After oversampling:  majority={len(majority)}, minority={len(minority_oversampled)}')
print(f'\nNew class balance:')
print(train_oversampled['is_fraud'].value_counts())

**Expected output:**
```
Before oversampling: majority=760, minority=40
After oversampling:  majority=760, minority=760

New class balance:
0    760
1    760
```

In [ ]:
X_train_over = train_oversampled[['amount', 'num_locations_24h']]
y_train_over = train_oversampled['is_fraud']

model_over = LogisticRegression()
model_over.fit(X_train_over, y_train_over)
preds_over = model_over.predict(X_test)

print('=== After Oversampling ===')
print(f'Accuracy:  {accuracy_score(y_test, preds_over):.4f}')
print(f'Precision: {precision_score(y_test, preds_over):.4f}')
print(f'Recall:    {recall_score(y_test, preds_over):.4f}')
print(f'F1 Score:  {f1_score(y_test, preds_over):.4f}')

**Expected output (approximate):**
```
=== After Oversampling ===
Accuracy:  0.9650
Precision: 0.7692
Recall:    1.0000
F1 Score:  0.8696
```

**Observation:** After oversampling, Recall went up to 1.00 — the model now catches every fraud case in the test set — but Precision dropped compared to the baseline. This makes sense once I thought about it: by force-balancing the training data, the model is now much more willing to flag something as fraud, which catches more real fraud but also produces more false alarms. Whether this tradeoff is worth it depends entirely on the actual cost of each error type — for fraud detection, missing a real fraud case is probably worse than a false alarm, so this tradeoff might actually be desirable here.

## Step 5: Undersampling the Majority Class

In [ ]:
majority_undersampled = majority.sample(n=len(minority), random_state=42)
train_undersampled = pd.concat([majority_undersampled, minority])

print(f'After undersampling: majority={len(majority_undersampled)}, minority={len(minority)}')
print(f'Total training rows: {len(train_undersampled)}  (down from {len(train_df)})')

X_train_under = train_undersampled[['amount', 'num_locations_24h']]
y_train_under = train_undersampled['is_fraud']

model_under = LogisticRegression()
model_under.fit(X_train_under, y_train_under)
preds_under = model_under.predict(X_test)

print('\n=== After Undersampling ===')
print(f'Accuracy:  {accuracy_score(y_test, preds_under):.4f}')
print(f'Precision: {precision_score(y_test, preds_under):.4f}')
print(f'Recall:    {recall_score(y_test, preds_under):.4f}')
print(f'F1 Score:  {f1_score(y_test, preds_under):.4f}')

**Expected output (approximate):**
```
After undersampling: majority=40, minority=40
Total training rows: 80  (down from 800)

=== After Undersampling ===
Accuracy:  0.9600
Precision: 0.8000
Recall:    0.8000
F1 Score:  0.8000
```

**Observation:** Undersampling threw away 720 majority-class rows to balance the dataset, going from 800 training rows down to just 80. The metrics came out roughly similar to oversampling here, but I'm a bit wary of this approach in general — discarding that much data feels wasteful, especially on a dataset that isn't huge to begin with. Oversampling at least keeps every original data point.

## Step 6: class_weight — Rebalancing Without Touching the Data

In [ ]:
# class_weight='balanced' automatically weights classes inversely
# to their frequency, without resampling anything

model_weighted = LogisticRegression(class_weight='balanced')
model_weighted.fit(X_train, y_train)
preds_weighted = model_weighted.predict(X_test)

print('=== With class_weight="balanced" ===')
print(f'Accuracy:  {accuracy_score(y_test, preds_weighted):.4f}')
print(f'Precision: {precision_score(y_test, preds_weighted):.4f}')
print(f'Recall:    {recall_score(y_test, preds_weighted):.4f}')
print(f'F1 Score:  {f1_score(y_test, preds_weighted):.4f}')

**Expected output (approximate):**
```
=== With class_weight="balanced" ===
Accuracy:  0.9650
Precision: 0.7778
Recall:    1.0000
F1 Score:  0.8750
```

**Observation:** This gave very similar results to oversampling, but without actually duplicating any rows or throwing any away. From my understanding, `class_weight='balanced'` works by changing how much each misclassified example penalises the loss function during training — a wrong prediction on a minority-class example costs more than a wrong prediction on a majority-class one. This feels like the most convenient option of the three, since it doesn't require modifying the dataset at all — just a single parameter change.

## Step 7: All Four Approaches Side by Side

In [ ]:
comparison = pd.DataFrame({
    'Approach':  ['Baseline (no balancing)', 'Oversampling', 'Undersampling', 'class_weight=balanced'],
    'Accuracy':  [accuracy_score(y_test, preds_baseline), accuracy_score(y_test, preds_over),
                  accuracy_score(y_test, preds_under), accuracy_score(y_test, preds_weighted)],
    'Precision': [precision_score(y_test, preds_baseline), precision_score(y_test, preds_over),
                  precision_score(y_test, preds_under), precision_score(y_test, preds_weighted)],
    'Recall':    [recall_score(y_test, preds_baseline), recall_score(y_test, preds_over),
                  recall_score(y_test, preds_under), recall_score(y_test, preds_weighted)],
    'F1':        [f1_score(y_test, preds_baseline), f1_score(y_test, preds_over),
                  f1_score(y_test, preds_under), f1_score(y_test, preds_weighted)]
})
print(comparison.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(comparison))
width = 0.2
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
colors = ['#1F3864', '#C00000', '#2CA02C', '#FFA500']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, comparison[metric], width, label=metric, color=color, alpha=0.85)
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(comparison['Approach'], rotation=15, fontsize=8.5)
ax.set_ylim(0.6, 1.05)
ax.legend(fontsize=9)
ax.set_title('Comparing Imbalance-Handling Strategies')
plt.tight_layout()
plt.savefig('balancing_comparison.png', dpi=150)
plt.show()

**Observation:** Looking at all four side by side, accuracy barely moves between approaches (they're all clustered near 0.95-0.98), but Recall tells a much more interesting story — it jumps from 0.80 in the baseline up to 1.00 for both oversampling and class_weight. This is the clearest illustration in this whole notebook of why accuracy alone is the wrong metric to optimise for on imbalanced data — it stays roughly flat across very different model behaviours, while the metric that actually matters for this problem (catching fraud) changes dramatically.

---

## Summary

| Approach | What it does | Tradeoff |
|---|---|---|
| Do nothing | Train on imbalanced data as-is | Accuracy looks fine, model may ignore the minority class entirely |
| Oversampling | Duplicate minority class rows | Keeps all real data, but can lead to overfitting on repeated examples |
| Undersampling | Discard majority class rows | Loses real data, especially harmful on smaller datasets |
| `class_weight='balanced'` | Penalise minority misclassifications more during training | No data modification needed, easy to apply |

## Personal Takeaway

The accuracy trap demo is the single most useful thing I built this week. Before this notebook, I would have looked at a 95% accuracy score and assumed the model was working well, full stop. Now I know to immediately check the class distribution first, and to look at Precision/Recall/F1 rather than trusting accuracy on its own — especially for anything where one outcome is rare but important (fraud, disease detection, and probably some aspect of the Spaceship Titanic target once I actually check its balance in Week 7). `class_weight='balanced'` is probably going to be my default first thing to try, since it's a one-parameter change rather than a whole resampling pipeline.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*